# Explorer les tables NeonDB

Objectif : se connecter à NeonDB depuis le `.env`, vérifier les tables disponibles, puis afficher un aperçu (`head`) de chaque table du projet Foot Predictor.

Le notebook ne modifie aucune donnée : il fait uniquement des lectures SQL.


## 1. Setup

Le notebook cherche une URL de connexion dans `NEON_DATABASE_URL`, `DATABASE_URL` ou `POSTGRES_URL`.


In [1]:
from pathlib import Path
import os

import pandas as pd
from sqlalchemy import create_engine, text
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 50)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / ".env").exists():
    raise FileNotFoundError("Fichier .env introuvable dans foot-predictor.")

SCHEMA = "public"
TABLES = ["competition", "team", "player", "match", "match_team", "lineup", "lineup_audit"]
HEAD_ROWS = 5


## 2. Charger la connexion Neon

On lit le `.env` sans afficher la valeur de connexion.


In [2]:
def load_env_file(path: Path) -> dict[str, str]:
    values = {}
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


def sqlalchemy_url(url: str) -> str:
    if url.startswith("postgres://"):
        return url.replace("postgres://", "postgresql+psycopg2://", 1)
    if url.startswith("postgresql://"):
        return url.replace("postgresql://", "postgresql+psycopg2://", 1)
    return url

local_env = load_env_file(PROJECT_ROOT / ".env")
DATABASE_URL = (
    os.environ.get("NEON_DATABASE_URL")
    or os.environ.get("DATABASE_URL")
    or os.environ.get("POSTGRES_URL")
    or local_env.get("NEON_DATABASE_URL")
    or local_env.get("DATABASE_URL")
    or local_env.get("POSTGRES_URL")
    or ""
)

if not DATABASE_URL:
    raise RuntimeError("Ajoute NEON_DATABASE_URL, DATABASE_URL ou POSTGRES_URL dans .env.")

engine = create_engine(sqlalchemy_url(DATABASE_URL), pool_pre_ping=True)
print("Connexion configuree :", bool(DATABASE_URL))


Connexion configuree : True


## 3. Résumé des tables

On vérifie le nombre de lignes et de colonnes avant les aperçus.


In [3]:
summary_rows = []

with engine.begin() as connection:
    for table_name in TABLES:
        row_count = connection.execute(
            text(f'SELECT COUNT(*) FROM "{SCHEMA}"."{table_name}"')
        ).scalar_one()
        column_count = connection.execute(
            text("""
                SELECT COUNT(*)
                FROM information_schema.columns
                WHERE table_schema = :schema
                  AND table_name = :table_name
            """),
            {"schema": SCHEMA, "table_name": table_name},
        ).scalar_one()
        summary_rows.append({"table": table_name, "rows": row_count, "columns": column_count})

tables_summary = pd.DataFrame(summary_rows)
display(tables_summary)


,table,rows,columns
0,competition,45,3
1,team,99,2
2,player,6705,127
3,match,4052,10
4,match_team,8104,7
5,lineup,171684,11
6,lineup_audit,4052,10


## 4. Aperçu de chaque table

Chaque bloc charge seulement quelques lignes avec `LIMIT`, pas toute la table.


In [4]:
heads = {}

for table_name in TABLES:
    query = f'SELECT * FROM "{SCHEMA}"."{table_name}" LIMIT {HEAD_ROWS}'
    heads[table_name] = pd.read_sql(query, engine)
    print(f"\n{table_name}")
    display(heads[table_name])



competition


,competition_id,competition_name,competition_type
0,CGB,CGB,None
1,KLUB,KLUB,None
2,POCP,POCP,None
3,A1,bundesliga,domestic_league
4,L1,bundesliga,domestic_league



team


,team_id,team_name
0,89,1.FC Union Berlin
1,5,AC Milan
2,197,AC Sparta Prague
3,430,ACF Fiorentina
4,2156,AEK Larnaca



player


...

,match_id,team_id,side,score,penalty_score,manager_name,formation
0,3573730,583,away,0,None,Mauricio Pochettino,4-3-3 Attacking
1,3573730,1082,home,1,None,Jocelyn Gourvennec,4-4-2 double 6
2,3575344,27,away,3,None,Julian Nagelsmann,4-2-3-1
3,3575344,16,home,1,None,Marco Rose,4-4-2 Diamond
4,3578075,281,away,0,None,Pep Guardiola,4-3-3 Attacking



lineup


,match_id,team_id,player_id,position_player,lineup_type,is_starting_match,shirt_number,team_captain,minute_start,minute_end,minutes_played
0,3573730,583,99343,Central Midfield,starting_lineup,1,21,0,0,90,90
1,3573730,583,536482,Central Midfield,starting_lineup,1,36,0,0,71,71
2,3573730,583,282041,Centre-Back,starting_lineup,1,3,1,0,81,81
3,3573730,583,228948,Centre-Back,starting_lineup,1,24,0,0,90,90
4,3573730,583,68863,Centre-Forward,starting_lineup,1,9,0,0,90,90



lineup_audit


,match_id,home_starting_count,away_starting_count,home_substitutes_count,away_substitutes_count,home_lineup_rows,away_lineup_rows,complete_starting_xi,has_bench_both_teams,complete_lineup_with_bench
0,3573730,11,11,9,9,20,20,1,1,1
1,3575344,11,11,9,9,20,20,1,1,1
2,3578075,11,11,9,9,20,20,1,1,1
3,3580217,11,11,7,7,18,18,1,1,1
4,3580326,11,11,7,7,18,18,1,1,1


## 5. Exemple de jointure "simple"

Petit aperçu optionnel pour vérifier que `lineup` rejoint bien `match`, `team` et `player`.


In [5]:
lineup_preview_query = f'''
SELECT
    l.match_id,
    m.match_date,
    ht.team_name AS home_team,
    at.team_name AS away_team,
    t.team_name AS lineup_team,
    mt.side,
    mt.score,
    mt.manager_name,
    mt.formation,
    l.lineup_type,
    l.position_player,
    l.minute_start,
    l.minute_end,
    l.minutes_played,
    l.player_id,
    p.player_name,
    p.sofifa_common_name,
    p.has_sofifa_profile
FROM "{SCHEMA}"."lineup" l
LEFT JOIN "{SCHEMA}"."match" m ON m.match_id = l.match_id
LEFT JOIN "{SCHEMA}"."match_team" mt ON mt.match_id = l.match_id AND mt.team_id = l.team_id
LEFT JOIN "{SCHEMA}"."team" t ON t.team_id = l.team_id
LEFT JOIN "{SCHEMA}"."match_team" hm ON hm.match_id = l.match_id AND hm.side = 'home'
LEFT JOIN "{SCHEMA}"."team" ht ON ht.team_id = hm.team_id
LEFT JOIN "{SCHEMA}"."match_team" am ON am.match_id = l.match_id AND am.side = 'away'
LEFT JOIN "{SCHEMA}"."team" at ON at.team_id = am.team_id
LEFT JOIN "{SCHEMA}"."player" p ON p.player_id = l.player_id
ORDER BY m.match_date DESC, l.match_id, mt.side, l.lineup_type, p.player_name
LIMIT {HEAD_ROWS * 4}
'''

display(pd.read_sql(lineup_preview_query, engine))

,match_id,match_date,home_team,away_team,lineup_team,side,score,manager_name,formation,lineup_type,position_player,minute_start,minute_end,minutes_played,player_id,player_name,sofifa_common_name,has_sofifa_profile
0,4626168,2026-05-24,Brighton & Hove Albion,Manchester United,Manchester United,away,3,Michael Carrick,4-2-3-1,starting_lineup,Right Winger,0.0,90.0,90,536835,Amad Diallo,Amad Diallo,1
1,4626168,2026-05-24,Brighton & Hove Albion,Manchester United,Manchester United,away,3,Michael Carrick,4-2-3-1,starting_lineup,Attacking Midfield,0.0,90.0,90,240306,Bruno Fernandes,Bruno Miguel Borges Fernandes,1
2,4626168,2026-05-24,Brighton & Hove Albion,Manchester United,Manchester United,away,3,Michael Carrick,4-2-3-1,starting_lineup,Centre-Forward,0.0,74.0,74,413039,Bryan Mbeumo,Bryan Mbeumo,1
3,4626168,2026-05-24,Brighton & Hove Albion,Manchester United,Manchester United,away,3,Michael Carrick,4-2-3-1,starting_lineup,Centre-Back,0.0,90.0,90,177907,Harry Maguire,Harry Maguire,1
4,4626168,2026-05-24,Brighton & Hove Albion,Manchester United,Manchester United,away,3,Michael Carrick,4-2-3-1,starting_lineup,Defensive Midfield,0.0,90.0,90,820374,Kobbie Mainoo,Kobbie Mainoo,1
5,4626168,2026-05-24,Brighton & Hove Albion,Manchester United,Manchester United,away,3,Michael Carrick,4-2-3-1,starting_lineup,Centre-Back,0.0,90.0,90,480762,Lisandro Martínez,Lisandro Martínez,1
6,4626168,2026-05-24,Brighton & Hove Albion,Manchester United,Manchester United,away,3,Michael Carrick,4-2-3-1,starting_lineup,Left-Back,0.0,82.0,82,183288,Luke Shaw,Luke Shaw,1
7,4626168,2026-05-24,Brighton & Hove Albion,Manchester United,Manchester United,away,3,Michael Carrick,4-2-3-1,starting_lineup,Central Midfield,0.0,74.0,74,346483,Mason Mount,Mason Mount,1
8,4626168,2026-05-24,Brighton & Hove Albion,Manchester United,Manchester United,away,3,Michael Carrick,4-2-3-1,starting_lineup,Right-Back,0.0,74.0,74,340456,Noussair Mazraoui,Noussair Mazraoui,1
9,4626168,2026-05-24,Brighton & Hove Albion,Manchester United,Manchester United,away,3,Michael Carrick,4-2-3-1,starting_lineup,Left Winger,0.0,62.0,62,926952,Patrick Dorgu,Patrick Dorgu,1


## Check des features descriptives des joueurs

In [ ]:
# Création d'un dataframe
players_df = pd.read_sql(f'SELECT * FROM "{SCHEMA}"."player"', engine)

print(players_df.shape)
display(players_df.head())